# Building a Robust Bash Tool for Amino-Acid Composition Analysis: Foundations First

Hello! If you're from a biology background and new to scripting, welcome. We'll build a simple tool to count amino acids (AAs) in protein sequences from FASTA files. Think of it like tallying residues in a sequence—useful for quick checks before deeper analysis, like spotting high-GC regions or hydrophobic cores.

We'll go **slowly**, one small idea at a time. Each step adds just **one new thing** (like a loop or a safety check), and we'll always test it works like before. No big jumps—by Step 15, you'll have a full script analyzing multiple files, and you'll understand basics: variables, loops, functions, conditionals, and error handling.

**Assumptions**: You have a `examples/` folder with 12 FASTA files (e.g., `seq_4krp_chainB.fasta`). If not, create one with sample data from PDB or UniProt. We're using a Bash kernel—run cells to see outputs.

Let's start with the simplest: a one-liner command. Breathe easy—we'll explain every bit.

## Step 1: The Simplest One-Liner – Count One Amino Acid

**Story So Far**: Nothing yet! We're just curious: How many Alanine ('A') residues are in one protein sequence?

**What We Do**: Use a chain of Unix commands to skip the FASTA header, flatten the sequence (remove newlines), find all 'A's, and count them. No script—just type and run.

**Why Start Here?**: It's like a lab quick-test: fast, shows the idea without code files. Biology folks often love pipelines like this for data wrangling—chain tools to process FASTA quickly.

**Key Ideas (Gentle Intro to Commands & Pipelines)**:
- `tail -n +2 file.fasta`: Like cutting off the top of a gel—skips line 1 (the >header line).
- `| tr -d '\n'`: Pipe (`|`) sends output to the next command; `tr` deletes newlines, making one long string (your full sequence, no breaks).
- `grep -o 'A'`: Searches for every 'A' (`grep` = global regular expression print) and prints each match on its own line (`-o` = only the match).
- `| wc -l`: Counts the lines (`wc -l` = word count, lines only)—each 'A' is one line, so that's your count!
- We store the flattened sequence in a variable `seq` (like a temporary lab note) to reuse it without re-running the pipeline.
- Variables: `$var` accesses the value; `$(command)` captures output into a var.

**Run This Cell**: It prints the count (should be 12 for our example). Play: Change 'A' to 'C' and rerun—what's the Cysteine count? This is the foundation—pipelines are Bash's superpower for bio data.

In [1]:
# Step 1: One-liner for Alanine count
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"  # Our example file (one protein chain)
seq=$(tail -n +2 "$file" | tr -d '\n')  # Flatten sequence (skip header), store in 'seq'
echo "$seq" | grep -o 'A' | wc -l  # Pipe seq to grep, count 'A's (output: 12)

      12


## Step 2: Add a For Loop – Count All 20 Amino Acids

**Story So Far**: Our one-liner counts one AA (e.g., 'A'). But proteins have 20 standard types—we want the full composition profile!

**What We Do**: Wrap the counting pipeline in a **for loop** to repeat it for each AA. First, list the 20 AAs in a variable.

**Why?**: Loops save repetitive typing—like pipetting the same volume for 20 samples instead of doing it one by one. No big changes; we reuse the exact flattening from Step 1, just loop the grep/wc part.

**Key Ideas (Intro to Loops & String Variables)**:
- `AAS="A C D E F G H I K L M N P Q R S T V W Y"`: A **string variable** with spaces separating items—like a comma-separated gene list, but spaces here.
- `for aa in $AAS; do ... done`: **For loop**—takes each item (`aa` is temporary name, like a loop variable) in the string, runs the inside code. It's like: "For each AA in the list, count it and print."
- `count=$(echo "$seq" | grep -o "$aa" | wc -l)`: Same pipeline, but `$aa` inserts the current AA (e.g., 'A' first time).
- `echo "$aa: $count"`: Prints a labeled result, like "A: 12". Quotes (`"`) around vars prevent issues if they had spaces (good habit).
- Loop runs 20 times—once per AA. We flatten `seq` only once (efficient!).

**Run This Cell**: Outputs 20 labeled lines (e.g., A: 12, C: 2, ..., Y: 11). Same file. Tip: Loops are everywhere in scripting—perfect for batching FASTA files or BLAST results later.

In [2]:
# Step 2: For loop over 20 AAs
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"
AAS="A C D E F G H I K L M N P Q R S T V W Y"  # String list of 20 standard AAs
seq=$(tail -n +2 "$file" | tr -d '\n')  # Same flattening as Step 1
for aa in $AAS; do  # For each AA in list...
  count=$(echo "$seq" | grep -o "$aa" | wc -l)  # ...count it with pipeline
  echo "$aa: $count"  # ...print labeled result
done  # End loop (20 lines output)

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 3: Make It a Real Script – Add Shebang & Save to File

**Story So Far**: Our loop works great in the notebook, but to share with a colleague or run from anywhere, we need a **script file** (like writing a protocol on paper).

**What We Do**: Save the Step 2 code as `step3.sh`, add a **shebang** line to tell the system "use Bash". Make it executable, then run like `./step3.sh`.

**Why?**: Scripts are reusable and portable—like a standard operating procedure (SOP) you email around the lab. The shebang makes it run directly, no need for "bash file.sh" every time.

**Key Ideas (Intro to Scripts & Files)**:
- `#!/usr/bin/env bash`: **Shebang**—first line, magic instruction: "Run this file with Bash" (`env` finds Bash on any system, like macOS or Linux).
- `cat > step3.sh << 'EOF' ... EOF`: Writes multi-line code to a file (`EOF` = end-of-file marker; quotes prevent var expansion during write).
- `chmod +x step3.sh`: **Change mode**—makes the file executable (+x = add execute permission), like unlocking a door.
- `./step3.sh`: Runs the script from the current directory (./ = here). Same output as Step 2—no changes inside the code!
- Pro tip: Edit scripts in a text editor (nano/vi/VS Code) for tweaks.

**Run This Cell**: Creates and runs `step3.sh`. Output: 20 AA lines. Now you have a file—try `ls step*.sh` to see it!

In [3]:
# Step 3: Save as script with shebang
cat > step3.sh << 'EOF'  # Write code to file
#!/usr/bin/env bash  # Shebang: Use Bash to run this
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"
AAS="A C D E F G H I K L M N P Q R S T V W Y"
seq=$(tail -n +2 "$file" | tr -d '\n')
for aa in $AAS; do
  count=$(echo "$seq" | grep -o "$aa" | wc -l)
  echo "$aa: $count"
done
EOF
chmod +x step3.sh  # Make executable (now you can run it directly)
./step3.sh  # Run it – same 20 lines as Step 2!

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 4: Add Safety Nets – Strict Mode for Errors

**Story So Far**: Our script runs fine, but what if the file is missing or a command fails quietly? It could give wrong zeros or hang. Time for **strict mode** to catch issues early.

**What We Do**: Add `set -euo pipefail` right after the shebang. This turns on safeguards without changing the logic.

**Why?**: Like double-checking reagents before an experiment—prevents cascading errors. In bio scripting, one bad FASTA shouldn't crash your whole analysis.

**Key Ideas (Intro to Error Handling & Flags)**:
- `set -e`: **Exit on error**—if any command fails (e.g., `tail` on missing file), stop the script immediately.
- `-u`: **Unset variables error**—if you typo `$file` as `$flie`, it complains instead of using empty.
- `-o pipefail`: **Pipe failures propagate**—in `grep | wc`, if `grep` fails silently, the whole pipe does too.
- Add once after shebang—applies to whole script. Output: Identical to Step 3.
- Test it: Edit `step4.sh` (change file path to junk), run—see it exit with error message.

**Run This Cell**: Creates `step4.sh`. Output: 20 lines. Safer foundation—essential for reliable bio tools!

In [4]:
# Step 4: Add strict mode safety
cat > step4.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail  # Safety first: -e exit on error, -u on unset vars, pipefail on pipes
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"
AAS="A C D E F G H I K L M N P Q R S T V W Y"
seq=$(tail -n +2 "$file" | tr -d '\n')
for aa in $AAS; do
  count=$(echo "$seq" | grep -o "$aa" | wc -l)
  echo "$aa: $count"
done
EOF
chmod +x step4.sh
./step4.sh  # Same output, but now catches errors (try bad file path!)

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 5: Safe Word Splitting – Set IFS & Switch to Arrays

**Story So Far**: Strict mode is on—good! But our AA list uses spaces, and loops might split weirdly on filenames with spaces (common in bio data, e.g., "human beta chain.fasta").

**What We Do**: Set `IFS` to control splitting, and upgrade `AAS` to a **Bash array** for safer looping.

**Why?**: Prevents loop disasters—like your for loop treating a spaced filename as two files. Arrays are cleaner for lists, like gene names or samples.

**Key Ideas (Intro to IFS & Arrays)**:
- `IFS=$'\n\t'`: **Internal Field Separator**—tells Bash to split words only on newlines/tabs, ignoring spaces. Safe for paths/sequences with spaces.
- `AAS=(A C D E F G H I K L M N P Q R S T V W Y)`: **Array literal**—() makes a list; no spaces needed inside.
- `for aa in "${AAS[@]}";`: Loops over array elements (`[@]` = all, quoted for safety). `${}` expands the var.
- Output: Same 20 lines. Arrays are a scripting staple—think looping over PDB IDs.

**Run This Cell**: `step5.sh`. Tip: Later, we'll use arrays for file lists too.

In [5]:
# Step 5: IFS for safe splitting & AAS as array
cat > step5.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'  # Split only on newlines/tabs—ignores spaces in filenames/seqs
AAS=(A C D E F G H I K L M N P Q R S T V W Y)  # Array: Safer list than space-string
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"
seq=$(tail -n +2 "$file" | tr -d '\n')
for aa in "${AAS[@]}"; do  # Loop over array elements (quoted!)
  count=$(echo "$seq" | grep -o "$aa" | wc -l)
  echo "$aa: $count"
done
EOF
chmod +x step5.sh
./step5.sh  # Identical output; handles spaced names now

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 6: First Function – Wrap the Counting Logic

**Story So Far**: Code is safe and looped, but all in one block—hard to reuse or test pieces. **Functions** let us bundle related code, like a sub-protocol.

**What We Do**: Create `count_aas()` function: Pass it the sequence, it loops and prints counts. Call it once at the end.

**Why?**: Modular—like a "Count Residues" section in your methods. Easy to call multiple times (e.g., for different chains) or test alone.

**Key Ideas (Intro to Functions & Arguments)**:
- `count_aas() { ... }`: **Function definition**—name, then body in braces {}. Like naming a recipe.
- `local seq="$1"`: Inside function, `$1` = first input (argument); `local` keeps it scoped (doesn't leak outside).
- The loop goes inside—function owns the counting.
- Call: `count_aas "$seq"`—passes seq as arg. Main script just preps and calls.
- Output: Same 20 lines. Functions build reusable tools—key for bio pipelines.

**Run This Cell**: `step6.sh`. Test: Add `count_aas "AAA"` after—counts 3 A's!

In [6]:
# Step 6: count_aas function
cat > step6.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
count_aas() {  # Function: Input seq, output counts
  local seq="$1"  # Arg 1 = seq; local = function-scope only
  for aa in "${AAS[@]}"; do
    count=$(echo "$seq" | grep -o "$aa" | wc -l)
    echo "$aa: $count"
  done
}
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"
seq=$(tail -n +2 "$file" | tr -d '\n')  # Prep seq
count_aas "$seq"  # Call: Pass seq to function
EOF
chmod +x step6.sh
./step6.sh  # Same output; now reusable function!

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 7: Second Function – Extract & Clean Sequence

**Story So Far**: Counting is modular—yay! Now, the sequence prep (flattening) is still inline. Let's function-ize it too for cleaner code.

**What We Do**: Define `get_seq()`: Takes a file, skips header, removes whitespace, outputs clean sequence. Use `awk` for robustness.

**Why?**: **Separation of concerns**—like "Sample Prep" vs. "Analysis" in a protocol. Easy to swap (e.g., add multi-FASTA support later).

**Key Ideas (Functions with Output & Awk Intro)**:
- `get_seq() { awk ... "$1"; }`: Function takes file arg (`$1`), uses `awk` (text processor, like grep on steroids).
- Awk script: `BEGIN{ORS=""}` no output newline; `/^>/{next}` skip headers; `gsub` remove spaces; `print` the rest.
- `seq=$(get_seq "$file")`: **Command substitution**—runs function, captures output as var.
- Main: Just calls functions. Output: Same.

**Run This Cell**: `step7.sh`. Awk might seem fancy, but it's common for FASTA—think of it as a smart text filter.

In [7]:
# Step 7: get_seq function
cat > step7.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
get_seq() {  # Function: Input file, output clean seq string
  awk 'BEGIN{ORS=""} /^>/{next} {gsub(/[[:space:]]+/,""); print}' "$1"  # Awk: Skip >, remove spaces
}
count_aas() {
  local seq="$1"
  for aa in "${AAS[@]}"; do
    count=$(echo "$seq" | grep -o "$aa" | wc -l)
    echo "$aa: $count"
  done
}
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"
seq=$(get_seq "$file")  # Capture function output
count_aas "$seq"
EOF
chmod +x step7.sh
./step7.sh  # Tidy: Prep -> Count; same results

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 8: Handle Edge Cases – Warn on Empty Sequences

**Story So Far**: Functions separate prep and count—elegant! But if a FASTA has no sequence (just header, or parse fail), we get all-zero counts. Not helpful.

**What We Do**: Add a simple check in `count_aas()`: If seq is empty, warn to stderr and skip printing zeros.

**Why?**: Real lab data has glitches (empty files, bad parses). A warning acts like a QC flag—alerts without crashing.

**Key Ideas (Intro to Conditionals & Stderr)**:
- `[[ -z "$seq" ]]`: **Test**: `-z` checks if string is empty (zero length). `[[ ]]` is Bash's safe if-like test.
- `&& { echo "WARN: Empty seq" >&2; return; }`: If true, run block: Print warning to **stderr** (`>&2` = error stream, doesn't mix with output), then `return` (exit function early).
- Place at function start. Output: Same for good file; test with `echo > empty.fasta` and edit file=.

**Run This Cell**: `step8.sh`. Conditionals are like if-statements—branch based on data, crucial for robust bio scripts.

In [8]:
# Step 8: Conditional check for empty seq
cat > step8.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
get_seq() {
  awk 'BEGIN{ORS=""} /^>/{next} {gsub(/[[:space:]]+/,""); print}' "$1"
}
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq from $file" >&2; return; }  # If empty, warn & skip
  for aa in "${AAS[@]}"; do
    count=$(echo "$seq" | grep -o "$aa" | wc -l)
    echo "$aa: $count"
  done
}
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"
seq=$(get_seq "$file")
count_aas "$seq"
EOF
chmod +x step8.sh
./step8.sh  # Warns on bad input; protects output

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 9: Flexible Input – Command-Line Arguments with Getopts

**Story So Far**: Single hardcoded file limits us. Let's make it take **-f file.fasta** as an option, with defaults and checks.

**What We Do**: Use `getopts` to parse flags. Add a usage function and validate the file exists/non-empty.

**Why?**: Scripts should adapt to your data—like choosing input based on experiment. Defaults make it easy to run without args.

**Key Ideas (Intro to Args & Parsing)**:
- `usage() { ... exit 2; }`: Simple function for help/error.
- `while getopts ":f:" opt; do`: **Getopts loop**—parses args; `:f:` means -f expects a value (file).
- `case $opt in f) file="$OPTARG" ;; \?) usage ;; esac`: Matches flag, sets var (`OPTARG` = value after flag).
- `[[ -f "$file" && -s "$file" ]]` : Test file exists (`-f`) and non-empty (`-s`); error if not, `exit 1` (general error code).
- Run: `./step9.sh` (default) or `./step9.sh -f other.fasta`.

**Run This Cell**: Uses default. Output: 20 lines. Args make scripts versatile—try your own FASTA!

In [9]:
# Step 9: Parse -f arg with getopts
cat > step9.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
usage() { echo "Usage: $0 [-f FILE]" >&2; exit 2; }  # Help function
file="examples/ex01_4krp_B/seq_4krp_chainB.fasta"  # Default file
while getopts ":f:" opt; do  # Parse loop: :f: = optional -f with value
  case $opt in
    f) file="$OPTARG" ;;  # Set file from arg
    \?) usage ;;  # Unknown flag
  esac
done
[[ -f "$file" && -s "$file" ]] || { echo "ERROR: Bad file: $file" >&2; exit 1; }  # Validate
get_seq() {
  awk 'BEGIN{ORS=""} /^>/{next} {gsub(/[[:space:]]+/,""); print}' "$1"
}
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq" >&2; return; }
  for aa in "${AAS[@]}"; do
    count=$(echo "$seq" | grep -o "$aa" | wc -l)
    echo "$aa: $count"
  done
}
seq=$(get_seq "$file")
count_aas "$seq"
EOF
chmod +x step9.sh
./step9.sh  # Runs with default; try -f yourfile.fasta

A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 10: Discover Files Automatically – Intro to Batch with Find

**Story So Far**: Args let us pick one file—flexible! But `examples/` has 12 FASTA files; we want to process all without listing manually.

**What We Do**: Use `find` to list matching files (`seq_*.fasta`), sort for consistency, then process just the first (prep for full loop next).

**Why?**: Automates folder scans—like grep-ing logs across a directory. Start small: Discover + log, then expand.

**Key Ideas (Intro to Find & Variables for Lists)**:
- `find "$dir" -type f -name 'seq_*.fasta' | sort`: **Find command**—searches dir for files (`-type f`), matching pattern (`-name`), pipes to `sort` for alphabetical order (reproducible).
- `files=$(find ...)`: Capture list as string var (newlines separate files).
- `first=$(echo "$files" | head -n1)`: First file (`head -1` = first line).
- `total=$(echo "$files" | wc -l)`: Count lines = total files.
- Drop -f arg for now (uses find). Output: Log + counts for first file.

**Run This Cell**: `step10.sh`. Foundational for batch bio: Find all .fasta, process en masse.

In [10]:
# Step 10: Auto-discover with find
cat > step10.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
INPUT_DIR="examples"  # Folder to scan
get_seq() {
  awk 'BEGIN{ORS=""} /^>/{next} {gsub(/[[:space:]]+/,""); print}' "$1"
}
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq" >&2; return; }
  for aa in "${AAS[@]}"; do
    count=$(echo "$seq" | grep -o "$aa" | wc -l)
    echo "$aa: $count"
  done
}
files=$(find "$INPUT_DIR" -type f -name 'seq_*.fasta' | sort)  # List & sort files
[[ -n "$files" ]] || { echo "ERROR: No FASTA files in $INPUT_DIR" >&2; exit 1; }
first=$(echo "$files" | head -n1)  # First file
total=$(echo "$files" | wc -l | tr -d ' ')  # Total (trim whitespace)
echo "Found $total files. Processing first: $first"
seq=$(get_seq "$first")
count_aas "$seq"
EOF
chmod +x step10.sh
./step10.sh  # Discovers batch; processes one—next: all!

Found 12 files. Processing first: examples/ex01_4krp_B/seq_4krp_chainB.fasta
A:       12
C:        2
D:        5
E:        7
F:        5
G:       14
H:        6
I:        3
K:        5
L:        7
M:        3
N:        7
P:        2
Q:        7
R:        6
S:       15
T:        7
V:        9
W:        3
Y:       11


## Step 11: Full Batch – Loop Over All Discovered Files

**Story So Far**: We can find and process one file from a batch. Now, loop over **all** files in the list.

**What We Do**: Use a while loop to read the file list into a loop var, process each with our functions, add blank lines between for readability. Drop the "first only" part.

**Why?**: True batching—like analyzing all chains in a PDB folder. While loops handle line-by-line input safely.

**Key Ideas (Intro to While Loops & Process Substitution)**:
- `while IFS= read -r f; do ... done < <(command)`: **While loop** for reading lines; `IFS=` preserves spaces, `-r` raw input. `< <(find ...)` = process substitution (runs find, feeds lines without subshell issues).
- Inside: `echo; seq=$(get_seq "$f"); count_aas ...; echo`: Process each file, blank separator.
- No array yet (while is portable). Output: 12 blocks (one per file).

**Run This Cell**: `step11.sh`. Now it scans everything—watch the progress!

In [11]:
# Step 11: While loop over all files
cat > step11.sh << 'EOF'
#!/usr/bin/env bash
# step11.sh — Full Batch: loop over all discovered files

set -euo pipefail
IFS=$'\n\t'

# Amino acids to count (order fixed for readability)
AAS=(A C D E F G H I K L M N P Q R S T V W Y)

INPUT_DIR="examples"   # adjust as needed

# get_seq: read FASTA, join all sequence lines, strip whitespace
get_seq() {
  awk '
    BEGIN{ RS=">"; FS="\n" }
    NR>1 {
      for(i=2;i<=NF;i++){ gsub(/[[:space:]]+/,"",$i); s=s $i }
    }
    END{ print s }
  ' "$1"
}

# count_aas: count occurrences of each AA in a given sequence
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq" >&2; return 0; }
  for aa in "${AAS[@]}"; do
    # use awk:gsub to count non-overlapping matches (O(1) pass in awk)
    local n
    n=$(awk -v s="$seq" -v aa="$aa" 'BEGIN{ print gsub(aa,"",s) }')
    printf "%-2s: %d\n" "$aa" "$n"
  done
}

# Main: loop safely over files discovered by find (null-delimited), sorted
# - IFS preserved; -r prevents backslash escapes; -d '' reads NUL separators
while IFS= read -r -d '' f; do
  echo "Processing: $f"
  seq=$(get_seq "$f")
  count_aas "$seq"
  echo                      # blank line between files
done < <(
  # find -> sort -z keeps NUL separators to be robust to spaces/newlines in names
  find "$INPUT_DIR" -type f -name 'seq_*.fasta' -print0 | sort -z
)
EOF
chmod +x step11.sh
set +u || true  # Notebook prep
./step11.sh  # Full batch: 12 proteins, each with 20 lines + blanks

Processing: examples/ex01_4krp_B/seq_4krp_chainB.fasta
A : 12
C : 2
D : 5
E : 7
F : 5
G : 14
H : 6
I : 3
K : 5
L : 7
M : 3
N : 7
P : 2
Q : 7
R : 6
S : 15
T : 7
V : 9
W : 3
Y : 11

Processing: examples/ex02_1ubq_A/seq_1ubq_chainA.fasta
A : 2
C : 0
D : 5
E : 6
F : 2
G : 6
H : 1
I : 7
K : 7
L : 9
M : 1
N : 2
P : 3
Q : 6
R : 4
S : 3
T : 7
V : 4
W : 0
Y : 1

Processing: examples/ex03_1crn_A/seq_1crn_chainA.fasta
A : 5
C : 6
D : 1
E : 1
F : 1
G : 4
H : 0
I : 5
K : 0
L : 1
M : 0
N : 3
P : 5
Q : 0
R : 2
S : 2
T : 6
V : 2
W : 0
Y : 2

Processing: examples/ex04_1yrf_A/seq_1yrf_chainA.fasta
A : 3
C : 0
D : 2
E : 2
F : 4
G : 2
H : 1
I : 0
K : 5
L : 5
M : 1
N : 1
P : 1
Q : 2
R : 1
S : 2
T : 1
V : 1
W : 1
Y : 0

Processing: examples/ex05_1qys_A/seq_1qys_chainA.fasta
A : 6
C : 0
D : 8
E : 9
F : 4
G : 8
H : 6
I : 8
K : 9
L : 7
M : 2
N : 6
P : 0
Q : 5
R : 3
S : 3
T : 9
V : 9
W : 0
Y : 4

Processing: examples/ex06_2n0a_A/seq_2n0a_chainA.fasta
A : 19
C : 0
D : 6
E : 18
F : 2
G : 18
H : 1
I : 2
K : 15
L :

## Step 12: Human-Readable Output – Extract Protein Names

**Story So Far**: Batch works, but output shows full paths (ugly). Let's pull friendly names like "seq_4krp_chainB".

**What We Do**: Add `name_from()` function using `basename` to strip path/extension. Print "Protein: $name" before counts.

**Why?**: Output should be lab-note friendly—not raw paths. Like labeling gels with short IDs.

**Key Ideas (Functions for Formatting & Basename)**:
- `name_from() { basename "$1" .fasta; }`: Takes full path, `basename` removes dir and .fasta suffix.
- In loop: `name=$(name_from "$f")`; `echo "Protein: $name"`.
- Output: Cleaner blocks, e.g., "Protein: seq_4krp_chainB" + 20 lines.

**Run This Cell**: `step12.sh`. Readability matters—your future self (or collab) thanks you!

In [12]:
# Step 12: name_from for labels
cat > step12.sh << 'EOF'
#!/usr/bin/env bash
# step12.sh — Human-Readable Output: extract protein names

set -euo pipefail
IFS=$'\n\t'

AAS=(A C D E F G H I K L M N P Q R S T V W Y)
INPUT_DIR="examples"

# Join FASTA sequence lines; strip whitespace
get_seq() {
  awk '
    BEGIN{ RS=">"; FS="\n" }
    NR>1 { for(i=2;i<=NF;i++){ gsub(/[[:space:]]+/,"",$i); s=s $i } }
    END{ print s }
  ' "$1"
}

# Derive short name without path or .fasta
name_from() {
  local base
  base=$(basename "$1" .fasta)
  printf "%s\n" "$base"
}

# Count AAs using awk gsub (single pass per AA)
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq" >&2; return 0; }
  for aa in "${AAS[@]}"; do
    local n
    n=$(awk -v s="$seq" -v aa="$aa" 'BEGIN{ print gsub(aa,"",s) }')
    printf "%-2s: %d\n" "$aa" "$n"
  done
}

# Stream files from find, sorted, NUL-safe
while IFS= read -r -d '' f; do
  name=$(name_from "$f")
  echo "Protein: $name"
  seq=$(get_seq "$f")
  count_aas "$seq"
  echo
done < <(find "$INPUT_DIR" -type f -name 'seq_*.fasta' -print0 | sort -z)
EOF
chmod +x step12.sh
set +u || true  # Notebook prep
./step12.sh  # Labeled proteins—easy to scan!

Protein: seq_4krp_chainB
A : 12
C : 2
D : 5
E : 7
F : 5
G : 14
H : 6
I : 3
K : 5
L : 7
M : 3
N : 7
P : 2
Q : 7
R : 6
S : 15
T : 7
V : 9
W : 3
Y : 11

Protein: seq_1ubq_chainA
A : 2
C : 0
D : 5
E : 6
F : 2
G : 6
H : 1
I : 7
K : 7
L : 9
M : 1
N : 2
P : 3
Q : 6
R : 4
S : 3
T : 7
V : 4
W : 0
Y : 1

Protein: seq_1crn_chainA
A : 5
C : 6
D : 1
E : 1
F : 1
G : 4
H : 0
I : 5
K : 0
L : 1
M : 0
N : 3
P : 5
Q : 0
R : 2
S : 2
T : 6
V : 2
W : 0
Y : 2

Protein: seq_1yrf_chainA
A : 3
C : 0
D : 2
E : 2
F : 4
G : 2
H : 1
I : 0
K : 5
L : 5
M : 1
N : 1
P : 1
Q : 2
R : 1
S : 2
T : 1
V : 1
W : 1
Y : 0

Protein: seq_1qys_chainA
A : 6
C : 0
D : 8
E : 9
F : 4
G : 8
H : 6
I : 8
K : 9
L : 7
M : 2
N : 6
P : 0
Q : 5
R : 3
S : 3
T : 9
V : 9
W : 0
Y : 4

Protein: seq_2n0a_chainA
A : 19
C : 0
D : 6
E : 18
F : 2
G : 18
H : 1
I : 2
K : 15
L : 4
M : 4
N : 3
P : 5
Q : 6
R : 0
S : 4
T : 10
V : 19
W : 0
Y : 4

Protein: seq_2zta_A+B
A : 2
C : 0
D : 2
E : 12
F : 0
G : 2
H : 2
I : 0
K : 10
L : 12
M : 2
N : 4
P : 0
Q : 2
R : 6

## Step 13: Optimize Counting – Switch to Awk for Efficiency

**Story So Far**: Output is readable, batch complete! But `grep | wc` spawns processes per AA (20x overhead). Use pure `awk` inside the function for speed.

**What We Do**: Replace grep/wc with `awk`'s `gsub` (global substitute)—counts matches by "replacing" them (but we don't need the replaced string).

**Why?**: Faster for large sequences (e.g., full proteomes). Single tool, no pipes—efficient like vectorized ops in R.

**Key Ideas (Awk for Counting)**:
- `count=$(awk -v s="$seq" -v aa="$aa" 'BEGIN{print gsub(aa, "", s)}')`: `-v` passes vars to awk; `gsub` counts substitutions (returns number, even 0); `BEGIN` runs before input.
- No echo/grep/wc—direct count. Output: Identical numbers.
- Awk is Bash's Swiss Army knife for text—learn it for FASTA/GFF parsing.

**Run This Cell**: `step13.sh`. Same output, but leaner—scales to big data.

In [13]:
# Step 13: Awk gsub for faster counts
cat > step13.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
INPUT_DIR="examples"
get_seq() {
  awk 'BEGIN{ORS=""} /^>/{next} {gsub(/[[:space:]]+/,""); print}' "$1"
}
name_from() { basename "$1" .fasta; }
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq" >&2; return; }
  for aa in "${AAS[@]}"; do
    local count  # Local var for count
    count=$(awk -v s="$seq" -v aa="$aa" 'BEGIN{print gsub(aa, "", s)}')  # Awk counts matches
    echo "$aa: $count"
  done
}
while IFS= read -r f; do
  echo "Protein: $(name_from "$f")"
  seq=$(get_seq "$f")
  count_aas "$seq"
  echo
done < <(find "$INPUT_DIR" -type f -name 'seq_*.fasta' | sort)
EOF
chmod +x step13.sh
set +u || true  # Notebook prep
./step13.sh  # Efficient counts; same output

Protein: seq_4krp_chainB
A: 12
C: 2
D: 5
E: 7
F: 5
G: 14
H: 6
I: 3
K: 5
L: 7
M: 3
N: 7
P: 2
Q: 7
R: 6
S: 15
T: 7
V: 9
W: 3
Y: 11

Protein: seq_1ubq_chainA
A: 2
C: 0
D: 5
E: 6
F: 2
G: 6
H: 1
I: 7
K: 7
L: 9
M: 1
N: 2
P: 3
Q: 6
R: 4
S: 3
T: 7
V: 4
W: 0
Y: 1

Protein: seq_1crn_chainA
A: 5
C: 6
D: 1
E: 1
F: 1
G: 4
H: 0
I: 5
K: 0
L: 1
M: 0
N: 3
P: 5
Q: 0
R: 2
S: 2
T: 6
V: 2
W: 0
Y: 2

Protein: seq_1yrf_chainA
A: 3
C: 0
D: 2
E: 2
F: 4
G: 2
H: 1
I: 0
K: 5
L: 5
M: 1
N: 1
P: 1
Q: 2
R: 1
S: 2
T: 1
V: 1
W: 1
Y: 0

Protein: seq_1qys_chainA
A: 6
C: 0
D: 8
E: 9
F: 4
G: 8
H: 6
I: 8
K: 9
L: 7
M: 2
N: 6
P: 0
Q: 5
R: 3
S: 3
T: 9
V: 9
W: 0
Y: 4

Protein: seq_2n0a_chainA
A: 19
C: 0
D: 6
E: 18
F: 2
G: 18
H: 1
I: 2
K: 15
L: 4
M: 4
N: 3
P: 5
Q: 6
R: 0
S: 4
T: 10
V: 19
W: 0
Y: 4

Protein: seq_2zta_A+B
A: 2
C: 0
D: 2
E: 12
F: 0
G: 2
H: 2
I: 0
K: 10
L: 12
M: 2
N: 4
P: 0
Q: 2
R: 6
S: 2
T: 0
V: 6
W: 0
Y: 2

Protein: seq_2zta_chainA
A: 1
C: 0
D: 1
E: 6
F: 0
G: 1
H: 1
I: 0
K: 5
L: 6
M: 1
N: 2
P: 0
Q: 1
R: 3
S: 1
T: 

## Step 14: Arrays for Files – Store List in Bash Array

**Story So Far**: While loop batches well, but arrays let us store/index the file list (e.g., for summaries later).

**What We Do**: Populate a `FILES` array from find, then for-loop over it. Portable—no mapfile needed.

**Why?**: Arrays enable extras like totals per AA across files. For-loop on arrays is simple/safe.

**Key Ideas (Arrays for Lists)**:
- `declare -a FILES=()`: Declare empty array.
- `while IFS= read -r f; do FILES+=("$f"); done < <(find ...)`: Append (`+=`) each file to array.
- `for f in "${FILES[@]}"; do ...`: Loop over all (quoted). Check `((${#FILES[@]} == 0))` for empty.
- Output: Same blocks.

**Run This Cell**: `step14.sh`. Arrays: Your go-to for sample lists or results.

In [14]:
# Step 14: File list as array
cat > step14.sh << 'EOF'
#!/usr/bin/env bash
set -euo pipefail
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
INPUT_DIR="examples"
get_seq() {
  awk 'BEGIN{ORS=""} /^>/{next} {gsub(/[[:space:]]+/,""); print}' "$1"
}
name_from() { basename "$1" .fasta; }
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq" >&2; return; }
  for aa in "${AAS[@]}"; do
    local count
    count=$(awk -v s="$seq" -v aa="$aa" 'BEGIN{print gsub(aa, "", s)}')
    echo "$aa: $count"
  done
}
declare -a FILES=()  # Empty array
while IFS= read -r f; do FILES+=("$f"); done < <(find "$INPUT_DIR" -type f -name 'seq_*.fasta' | sort)  # Fill array
((${#FILES[@]} == 0)) && { echo "ERROR: No files" >&2; exit 1; }  # Check empty
for f in "${FILES[@]}"; do  # For-loop over array
  echo "Protein: $(name_from "$f")"
  seq=$(get_seq "$f")
  count_aas "$seq"
  echo
done
EOF
chmod +x step14.sh
set +u || true  # Notebook prep
./step14.sh  # Array-powered batch; flexible for extensions

Protein: seq_4krp_chainB
A: 12
C: 2
D: 5
E: 7
F: 5
G: 14
H: 6
I: 3
K: 5
L: 7
M: 3
N: 7
P: 2
Q: 7
R: 6
S: 15
T: 7
V: 9
W: 3
Y: 11

Protein: seq_1ubq_chainA
A: 2
C: 0
D: 5
E: 6
F: 2
G: 6
H: 1
I: 7
K: 7
L: 9
M: 1
N: 2
P: 3
Q: 6
R: 4
S: 3
T: 7
V: 4
W: 0
Y: 1

Protein: seq_1crn_chainA
A: 5
C: 6
D: 1
E: 1
F: 1
G: 4
H: 0
I: 5
K: 0
L: 1
M: 0
N: 3
P: 5
Q: 0
R: 2
S: 2
T: 6
V: 2
W: 0
Y: 2

Protein: seq_1yrf_chainA
A: 3
C: 0
D: 2
E: 2
F: 4
G: 2
H: 1
I: 0
K: 5
L: 5
M: 1
N: 1
P: 1
Q: 2
R: 1
S: 2
T: 1
V: 1
W: 1
Y: 0

Protein: seq_1qys_chainA
A: 6
C: 0
D: 8
E: 9
F: 4
G: 8
H: 6
I: 8
K: 9
L: 7
M: 2
N: 6
P: 0
Q: 5
R: 3
S: 3
T: 9
V: 9
W: 0
Y: 4

Protein: seq_2n0a_chainA
A: 19
C: 0
D: 6
E: 18
F: 2
G: 18
H: 1
I: 2
K: 15
L: 4
M: 4
N: 3
P: 5
Q: 6
R: 0
S: 4
T: 10
V: 19
W: 0
Y: 4

Protein: seq_2zta_A+B
A: 2
C: 0
D: 2
E: 12
F: 0
G: 2
H: 2
I: 0
K: 10
L: 12
M: 2
N: 4
P: 0
Q: 2
R: 6
S: 2
T: 0
V: 6
W: 0
Y: 2

Protein: seq_2zta_chainA
A: 1
C: 0
D: 1
E: 6
F: 0
G: 1
H: 1
I: 0
K: 5
L: 6
M: 1
N: 2
P: 0
Q: 1
R: 3
S: 1
T: 

## Step 15: Notebook Safety – Handle Jupyter Quirks

**Story So Far**: Full batch with arrays—almost production-ready! But in Jupyter/Bash kernels, vars like `HISTTIMEFORMAT` can crash on exit.

**What We Do**: Add tweaks: Unset history var, trap to reset flags on exit. Ensures clean runs.

**Why?**: Notebooks are great for teaching, but kernels can be finicky. These make scripts portable (run in terminal too).

**Key Ideas (Traps & Environment Vars)**:
- `HISTTIMEFORMAT=''`: Clears history timestamp (avoids unbound errors).
- `trap 'set +u' EXIT`: **Trap**—runs command on exit (here, disable -u to prevent teardown crash).
- `set +u || true` before run: Neutralizes in notebook.
- Output: Same. Final polish!

**Run This Cell**: `step15.sh`. Safe for notebooks—your tool is robust now.

In [15]:
# Step 15: Jupyter safety tweaks
cat > step15.sh << 'EOF'
#!/usr/bin/env bash
HISTTIMEFORMAT=''  # Clear history var for notebooks
set -euo pipefail
trap 'set +u' EXIT  # On exit, disable -u to avoid kernel crash
IFS=$'\n\t'
AAS=(A C D E F G H I K L M N P Q R S T V W Y)
INPUT_DIR="examples"
get_seq() {
  awk 'BEGIN{ORS=""} /^>/{next} {gsub(/[[:space:]]+/,""); print}' "$1"
}
name_from() { basename "$1" .fasta; }
count_aas() {
  local seq="$1"
  [[ -z "$seq" ]] && { echo "WARN: Empty seq" >&2; return; }
  for aa in "${AAS[@]}"; do
    local count
    count=$(awk -v s="$seq" -v aa="$aa" 'BEGIN{print gsub(aa, "", s)}')
    echo "$aa: $count"
  done
}
declare -a FILES=()
while IFS= read -r f; do FILES+=("$f"); done < <(find "$INPUT_DIR" -type f -name 'seq_*.fasta' | sort)
((${#FILES[@]})) || { echo "ERROR: No files" >&2; exit 1; }
for f in "${FILES[@]}"; do
  echo "Protein: $(name_from "$f")"
  seq=$(get_seq "$f")
  count_aas "$seq"
  echo
done
EOF
chmod +x step15.sh
set +u || true  # Notebook prep
./step15.sh  # Full, safe batch output

Protein: seq_4krp_chainB
A: 12
C: 2
D: 5
E: 7
F: 5
G: 14
H: 6
I: 3
K: 5
L: 7
M: 3
N: 7
P: 2
Q: 7
R: 6
S: 15
T: 7
V: 9
W: 3
Y: 11

Protein: seq_1ubq_chainA
A: 2
C: 0
D: 5
E: 6
F: 2
G: 6
H: 1
I: 7
K: 7
L: 9
M: 1
N: 2
P: 3
Q: 6
R: 4
S: 3
T: 7
V: 4
W: 0
Y: 1

Protein: seq_1crn_chainA
A: 5
C: 6
D: 1
E: 1
F: 1
G: 4
H: 0
I: 5
K: 0
L: 1
M: 0
N: 3
P: 5
Q: 0
R: 2
S: 2
T: 6
V: 2
W: 0
Y: 2

Protein: seq_1yrf_chainA
A: 3
C: 0
D: 2
E: 2
F: 4
G: 2
H: 1
I: 0
K: 5
L: 5
M: 1
N: 1
P: 1
Q: 2
R: 1
S: 2
T: 1
V: 1
W: 1
Y: 0

Protein: seq_1qys_chainA
A: 6
C: 0
D: 8
E: 9
F: 4
G: 8
H: 6
I: 8
K: 9
L: 7
M: 2
N: 6
P: 0
Q: 5
R: 3
S: 3
T: 9
V: 9
W: 0
Y: 4

Protein: seq_2n0a_chainA
A: 19
C: 0
D: 6
E: 18
F: 2
G: 18
H: 1
I: 2
K: 15
L: 4
M: 4
N: 3
P: 5
Q: 6
R: 0
S: 4
T: 10
V: 19
W: 0
Y: 4

Protein: seq_2zta_A+B
A: 2
C: 0
D: 2
E: 12
F: 0
G: 2
H: 2
I: 0
K: 10
L: 12
M: 2
N: 4
P: 0
Q: 2
R: 6
S: 2
T: 0
V: 6
W: 0
Y: 2

Protein: seq_2zta_chainA
A: 1
C: 0
D: 1
E: 6
F: 0
G: 1
H: 1
I: 0
K: 5
L: 6
M: 1
N: 2
P: 0
Q: 1
R: 3
S: 1
T: 

## Next Steps & Extensions: Your Roadmap to Advanced Scripting

You've built a complete, robust tool—congrats! It scans folders, counts AAs, handles errors, and outputs cleanly. Now, extend it like a growing protocol. Each idea builds on Step 15—pick one, implement incrementally.

1. **Tabular Output (TSV/CSV)**: Print as table (protein | A | C | ... | Y); use `printf` for columns, redirect to `output.tsv`. Atomic write: Temp file + `mv`.

2. **Add Lengths & Percentages**: Compute `len=${#seq}` (string length); print `% = (count / len * 100) | bc -l` (use `bc` for math). Useful for composition plots.

3. **Grand Totals Summary**: Accumulate sums in arrays (e.g., `totalA[$aa]+=$count`); print final row. Optional separate summary file.

4. **Input Validation**: Check for non-standard chars (e.g., `grep -o '[BJOUXZ*]'`); warn or `--strict` to exit. Like alphabet QC.

5. **Gzip Support**: If `.gz`, use `zcat "$f" | get_seq` (modify `get_seq`). Streams—no unpack needed.

6. **Parallel Processing**: `export -f get_seq count_aas; printf '%s\n' "${FILES[@]}" | xargs -P4 -I {} bash -c '...'`. Sort output after for order.

7. **Full CLI**: Add `-i DIR -o FILE --percent --summary`; enhance `getopts`. Add `--help` with usage.

8. **Unit Tests**: Use `bats` framework; create test FASTA, assert outputs match goldens.

9. **Benchmarking**: `time ./step15.sh`; compare grep vs awk on large seq (e.g., 10k AAs).

10. **Integration**: Wrap in Snakemake (rules for input/output); Dockerize with `FROM ubuntu:22.04` + tools.

